In [1]:
import tensorflow as tf
from pathlib import Path
import time
import pandas as pd
PROJECT_DIR = Path.cwd().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd()
MODELS = {
    "custom_cnn": PROJECT_DIR/"models"/"custom_cnn"/"best_model.h5",
    "transfer_learning": PROJECT_DIR/"models"/"transfer_learning"/"best_model.h5"
}
IMG_SIZE = (224,224)
BATCH_SIZE = 16

from tensorflow.keras.preprocessing import image_dataset_from_directory
test_ds = image_dataset_from_directory(PROJECT_DIR/"data"/"classification_dataset"/"test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False)
normalization = tf.keras.layers.Rescaling(1./255)
test_ds = test_ds.map(lambda x,y: (normalization(x), y)).cache().prefetch(tf.data.AUTOTUNE)


Found 100 files belonging to 2 classes.


In [2]:
def evaluate_inference(model_path, ds, n_batches=10):
    model = tf.keras.models.load_model(model_path)
    params = model.count_params()
    # warmup
    for i, (x,y) in enumerate(ds.take(2)):
        _ = model.predict(x)
    # timed runs
    times = []
    i=0
    for x,y in ds.take(n_batches):
        start = time.time()
        _ = model.predict(x)
        end = time.time()
        times.append(end-start)
        i+=1
    avg_time_per_batch = sum(times)/len(times)
    avg_time_per_image = avg_time_per_batch / x.shape[0]
    return {"params":params, "avg_batch_time":avg_time_per_batch, "avg_time_per_image":avg_time_per_image}

rows=[]
for name, path in MODELS.items():
    if path.exists():
        stats = evaluate_inference(path, test_ds, n_batches=8)
        rows.append((name, stats))
    else:
        rows.append((name, None))
pd.DataFrame({r[0]:r[1] for r in rows}).T


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 552ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 513ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 553ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 543ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 535ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 542ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 547ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 547ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


,params,avg_batch_time,avg_time_per_image
custom_cnn,111426.0,0.277918,0.069479
transfer_learning,4412197.0,1.352925,0.338231


In [3]:
df = pd.DataFrame({r[0]:r[1] for r in rows}).T
df.to_csv(PROJECT_DIR/"outputs"/"model_comparison"/"final_comparison.csv")
display(df)


,params,avg_batch_time,avg_time_per_image
custom_cnn,111426.0,0.277918,0.069479
transfer_learning,4412197.0,1.352925,0.338231
